## Combining the Data
For the selected seasons (2018-2019 until 2026-2027) 

Last Updated: Nov.10, 2026

In [1]:
from pathlib import Path
import pandas as pd

In [2]:
root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent

files = sorted((root/'data'/'processed').glob('games_*.parquet'))

# Should have 8 (season) files (as of the updated date)
assert len(files) == 8, "Unexpected number of files."

# Combining the files
games_all = pd.concat(
    [pd.read_parquet(file) for file in files],
    ignore_index=True
)

# Sorting the games by date and game_id
games_all = games_all.sort_values(by=['game_date', 'game_id']).reset_index(drop=True)

display(games_all.head())

,game_id,game_date,season,home_id,away_id,home_team,away_team,home_score,away_score,home_win
0,401070213,2018-10-16,2019,2,20,BOS,PHI,105,87,1
1,401070214,2018-10-16,2019,9,25,GS,OKC,108,100,1
2,401070215,2018-10-17,2019,10,3,HOU,NO,112,131,0
3,401070216,2018-10-17,2019,21,6,PHX,DAL,121,100,1
4,401070693,2018-10-17,2019,30,15,CHA,MIL,112,113,0


## Validating the final dataset


In [3]:
# All games have their own unique game_id
assert games_all['game_id'].is_unique
# No null games
assert not games_all.isna().any().any()
# home_win values are either 0 (loss) or 1 (win)
assert games_all['home_win'].isin([0,1]).all()

summary = (
    games_all.groupby('season').agg(
        games=('game_id', 'count'),
        home_win_rate=('home_win', 'mean'),
        teams=('home_id', 'nunique')
    )
)

display(summary)



,games,home_win_rate,teams
season,,,
2019,1230,0.592683,30
2020,1059,0.551464,30
2021,1080,0.543519,30
2022,1230,0.543902,30
2023,1230,0.580488,30
2024,1230,0.543089,30
2025,1230,0.543902,30
2026,1230,0.554472,30


In a primary inspection, it was noticed that in the teams column, every row has more than 30 teams even though NBA has 30 teams total. There seemed to be all-star teams included in the regular season data. 

Upon further inspection, regular season games have `type_abbreviation` of `STD`. The `clean_games.py` script will be updated accordingly to filter out the all-star games and the NBA Cup Championship games (`ALLSTAR` and `CC`, respectively). 

However, in the earlier seasons of 2018-2019, and 2019-2020, the all star game and the rising stars games were also labeled as `STD`. Thus, as an additional safeguard, `clean_games.py` will also be updated to filter only for the 30 existing NBA teams. 

## Saving the combined data

In [5]:
games_all.to_parquet(root/'data'/'processed'/'games_all.parquet', index=False)